# vLLM Zero-Shot and Few-Shot Prompting

This notebook adapts the vLLM prompting experiments from my Master's thesis on **long-term cardiovascular mortality prediction**. It preserves the original model configuration, batched inference, response parsing and ROC-AUC evaluation structure.

The original ATTICA dataset and private University of Luxembourg dependencies have been replaced with **synthetic examples and independently implemented prompts** using the same twelve features. The resulting scores do not reproduce the thesis findings.

**Execution:** Requires a compatible vLLM/Transformers environment and GPU. The model can be changed through `model_name`. This public adaptation has not been GPU-tested.

In [ ]:
# Parameters
model_name = "google/medgemma-27b-text-it"
model_label = "MedGemma"

n_positive = 10
n_negative = 10
random_seed = 42

run_zeroshot = True
run_fewshot = True

use_wandb = False
wandb_project = "attica-vllm"
wandb_run_name = None


# vLLM inference parameters
sampling_temperature = 0.0
sampling_max_tokens = 80
sampling_top_p = 1.0
use_chat_template = True
batch_size = 10



In [ ]:
load_model = run_zeroshot or run_fewshot

print(f"Model label: {model_label}")
print(f"Model name: {model_name}")
print(f"Run zero-shot: {run_zeroshot}")
print(f"Run few-shot: {run_fewshot}")
print(f"Load model: {load_model}")
print(f"Sampling temperature: {sampling_temperature}")
print(f"Sampling max tokens: {sampling_max_tokens}")
print(f"Sampling top-p: {sampling_top_p}")
print(f"Use chat template: {use_chat_template}")
print(f"Batch size: {batch_size}")

if wandb_run_name is None:
    wandb_run_name = f"{model_label}_ATTICA_{n_positive}_{n_negative}"

if use_wandb:
    import wandb

    wandb.init(
        project=wandb_project,
        name=wandb_run_name,
        config={
            "model_name": model_name,
            "model_label": model_label,
            "n_positive": n_positive,
            "n_negative": n_negative,
            "random_seed": random_seed,
            "run_zeroshot": run_zeroshot,
            "run_fewshot": run_fewshot,
            "sampling_temperature": sampling_temperature,
            "sampling_max_tokens": sampling_max_tokens,
            "sampling_top_p": sampling_top_p,
            "use_chat_template": use_chat_template,
            "batch_size": batch_size,
            "features": "synthetic ATTICA_12-style",
            "labels": "synthetic_binary",
            "cohort": "synthetic_demo",
        },
    )
else:
    print("wandb logging disabled.")

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

## Synthetic inputs replacing private ATTICA dependencies

Only this data-loading/prompt-generation step is independently reimplemented. The generated records deliberately resemble the **feature schema**, not the original cohort or its distribution. Few-shot examples are fixed fictional examples, separate from evaluation rows. The toy labels are assigned by an invented rule and are not cardiovascular outcomes.


In [ ]:
# Public replacement for the private cohort loader and get_prompts_ds.
# Synthetic values are invented solely to make the inference workflow inspectable.
FEATURES_12 = ["age", "sex", "LDL", "HDL", "TGL", "DM", "smoking_ever",
               "MedDietScore", "creatinine", "CRP", "Fibrinogen", "Hcy"]
rng = np.random.default_rng(random_seed)
size = n_positive + n_negative
COHORT = pd.DataFrame({
    "age": rng.integers(25, 85, size),
    "sex": rng.choice(["female", "male"], size=size),
    "LDL": rng.uniform(70, 180, size).round(1),
    "HDL": rng.uniform(30, 85, size).round(1),
    "TGL": rng.uniform(55, 225, size).round(1),
    "DM": rng.choice(["no", "yes"], size=size),
    "smoking_ever": rng.choice(["no", "yes"], size=size),
    "MedDietScore": rng.integers(10, 55, size),
    "creatinine": rng.uniform(0.5, 1.7, size).round(2),
    "CRP": rng.uniform(0.2, 8.0, size).round(2),
    "Fibrinogen": rng.uniform(180, 450, size).round(1),
    "Hcy": rng.uniform(5, 22, size).round(1),
})
# Fictional labels assigned to preserve the requested sample counts.
# No ground-truth ATTICA measurements or original label-generation logic used.
labels = pd.Series([1] * n_positive + [0] * n_negative, dtype=int)
permutation = rng.permutation(size)
COHORT = COHORT.iloc[permutation].reset_index(drop=True)
labels = labels.iloc[permutation].reset_index(drop=True)


def describe_patient(patient):
    return "; ".join(f"{key}: {patient[key]}" for key in FEATURES_12)


TASK = (
    "For this synthetic demonstration, provide a percentage score for the positive "
    "class based on the listed fictional features. Answer in the format Risk: NN%. "
    "This is not a clinical assessment."
)
FEW_SHOT_EXAMPLES = [
    ("age: 30; sex: female; LDL: 85; HDL: 65; TGL: 80", "Risk: 10%"),
    ("age: 76; sex: male; LDL: 165; HDL: 35; TGL: 200", "Risk: 90%"),
]


def make_prompt(patient, fewshot=False):
    examples = ""
    if fewshot:
        examples = "\n\n".join(
            f"Fictional example: {description}\nAnswer: {answer}"
            for description, answer in FEW_SHOT_EXAMPLES
        ) + "\n\n"
    return f"{TASK}\n\n{examples}Features: {describe_patient(patient)}\nAnswer:"


# Keep the original dataset field names ('oneshot', 'fewshot') used below.
class PromptDataset:
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        return self.records[index]
    def select(self, indices):
        return [self.records[i] for i in indices]


dataset = PromptDataset([
    {"oneshot": make_prompt(row), "fewshot": make_prompt(row, fewshot=True)}
    for row in COHORT.to_dict("records")
])
pattern = r'(\d+\.?\d*)\s*%?'
print(f"Synthetic inputs: {len(dataset)} ({n_positive} positive, {n_negative} negative)")
print("No ATTICA participant data loaded.")


## vLLM inference and evaluation

The following sections preserve the original notebook's tokenizer, vLLM configuration, test calls, batched generation, parsing, and ROC-AUC computation, with **participant-level printing and CSV exports removed**. Note: because the inputs and labels are synthetic, the resulting AUC has no clinical or thesis interpretation.


In [ ]:
if load_model:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    def format_prompt(prompt):
        if use_chat_template:
            messages = [{"role": "user", "content": prompt}]
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        return prompt
else:
    print("Skipping tokenizer setup because both zero-shot and few-shot runs are disabled.")


In [ ]:
if load_model:
    from vllm import LLM, SamplingParams

    sampling_params = SamplingParams(
        temperature=sampling_temperature,
        max_tokens=sampling_max_tokens,
        top_p=sampling_top_p,
    )

    llm = LLM(
        model=model_name,
        trust_remote_code=True,
    )
else:
    print("Skipping vLLM model loading because both zero-shot and few-shot runs are disabled.")


In [ ]:
if load_model and run_zeroshot:
    test_prompt = format_prompt(dataset[0]["oneshot"])  # internally still named "oneshot", but this is zero-shot
    test_output = llm.generate([test_prompt], sampling_params)[0]

    print("Test generation completed (output omitted).")
else:
    print("Skipping zero-shot test generation.")

In [ ]:
if load_model and run_fewshot:
    fewshot_test_prompt = format_prompt(dataset[0]["fewshot"])
    fewshot_test_output = llm.generate([fewshot_test_prompt], sampling_params)[0]

    print("Test generation completed (output omitted).")
else:
    print("Skipping few-shot test generation.")

In [ ]:
if run_zeroshot:
    y_true = np.asarray(labels.array, dtype=int)
    y_pred, texts = [], []
    for i in tqdm(range(0, len(dataset), batch_size)):
        batch = range(i, min(i + batch_size, len(dataset)))
        prompts = [format_prompt(patient["oneshot"]) for patient in dataset.select(batch)]
        outputs = llm.generate(prompts, sampling_params)
        for output in outputs:
            text = output.outputs[0].text
            texts.append(text)
            m = re.search(pattern, text)
            y_pred.append(float(m.group(1)) if m and 0 <= float(m.group(1)) <= 100 else np.nan)
    y_pred = np.array(y_pred, dtype=float) / 100
    valid = ~np.isnan(y_pred)
    print("NaNs:", int((~valid).sum()), "out of", len(y_pred))

    if valid.sum() and len(set(y_true[valid])) >= 2:
        auc = roc_auc_score(y_true[valid], y_pred[valid])
        print(f"\nGENERATE vLLM BATCH {model_label} ZEROSHOT AUC: {auc:.3f}  (valid {int(valid.sum())}/{len(y_pred)})\n")
        if use_wandb:
            wandb.log({"zeroshot_auc": auc, "zeroshot_valid_predictions": int(valid.sum())})
    else:
        print("ZEROSHOT AUC not computable (no valid preds or single class).")
else:
    print("Skipping zero-shot run.")

In [ ]:
if run_fewshot:
    y_true = np.asarray(labels.array, dtype=int)
    y_pred, texts = [], []
    for i in tqdm(range(0, len(dataset), batch_size)):
        batch = range(i, min(i + batch_size, len(dataset)))
        prompts = [format_prompt(patient["fewshot"]) for patient in dataset.select(batch)]
        outputs = llm.generate(prompts, sampling_params)
        for output in outputs:
            text = output.outputs[0].text
            texts.append(text)
            m = re.search(pattern, text)
            y_pred.append(float(m.group(1)) if m and 0 <= float(m.group(1)) <= 100 else np.nan)
    y_pred = np.array(y_pred, dtype=float) / 100
    valid = ~np.isnan(y_pred)
    print("NaNs:", int((~valid).sum()), "out of", len(y_pred))

    if valid.sum() and len(set(y_true[valid])) >= 2:
        auc = roc_auc_score(y_true[valid], y_pred[valid])
        print(f"\nGENERATE vLLM BATCH {model_label} FEWSHOT AUC: {auc:.3f}  (valid {int(valid.sum())}/{len(y_pred)})\n")
        if use_wandb:
            wandb.log({"fewshot_auc": auc, "fewshot_valid_predictions": int(valid.sum())})
    else:
        print("FEWSHOT AUC not computable (no valid preds or single class).")
else:
    print("Skipping few-shot run.")


In [ ]:
if use_wandb:
    wandb.finish()